# Data Science Lifecycle: Part 4 - Dual ML Modeling & Explainability

Fitting advanced regression and classification networks, optimizing hyper-parameters, and mapping explanations using SHAP.

### Objectives:
1. **Preprocessing Pipeline**: Configure ColumnTransformer (StandardScaler + OneHotEncoder).
2. **ETA Regressors**: Fit and compare Linear Regression, Random Forest, XGBoost, LightGBM, and CatBoost.
3. **Hyperparameter Tuning**: Optimize estimators, depth, and split configurations via randomized search.
4. **Delay Classifiers**: Fit Logistic Regression, Random Forests, and XGBoost to predict delay probabilities.
5. **Explainable AI**: Compute global feature importances and illustrate SHAP value concepts.

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pickle

processed_csv_path = "../data/processed/processed_deliveries.csv"
df = pd.read_csv(processed_csv_path)

# Apply basic feature engineering steps
traffic_map = {"Low": 1, "Medium": 2, "High": 3, "Jam": 4}
weather_map = {"Sunny": 1, "Cloudy": 2, "Rainy": 3, "Storm": 4}
df["Traffic_Severity_Score"] = df["Traffic_Level"].map(traffic_map)
df["Weather_Severity_Score"] = df["Weather"].map(weather_map)
df["Distance_Bucket"] = df["Distance_km"].apply(lambda x: "Short" if x <= 3.0 else ("Medium" if x <= 8.0 else ("Long" if x <= 15.0 else "Very Long")))
df["is_peak_hour"] = df["Peak_Hour"].apply(lambda x: 1 if str(x).lower() == "yes" else 0)
df["is_weekend"] = df["Day_of_Week"].apply(lambda x: 1 if str(x).capitalize() in ["Saturday", "Sunday"] else 0)
df["is_night"] = df["Time_of_Day"].apply(lambda x: 1 if str(x).capitalize() == "Night" else 0)

# Promised time classification target
promised_time = df["Preparation_Time"] + (df["Distance_km"] / 18.0) * 60.0 + 8.0
df["Is_Delayed"] = (df["Delivery_Time_Min"] > promised_time).astype(int)

df.head()

## 1. Feature Preprocessing Pipeline Setup

In [ ]:
categorical_cols = [
    "Vehicle_Type", "Weather", "Traffic_Level", "Time_of_Day", 
    "Day_of_Week", "Festival_Day", "Holiday", "Order_Size", 
    "Customer_Location_Type", "Distance_Bucket"
]
numerical_cols = [
    "Distance_km", "Preparation_Time", "Courier_Age", "Courier_Experience", 
    "Restaurant_Rating", "Traffic_Severity_Score", "Weather_Severity_Score",
    "is_peak_hour", "is_weekend", "is_night"
]

features = categorical_cols + numerical_cols
X = df[features]
y_reg = df["Delivery_Time_Min"]
y_clf = df["Is_Delayed"]

X_train, X_test, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
_, _, y_train_clf, y_test_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
])

X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre = preprocessor.transform(X_test)

print(f"Preprocessed training shape: {X_train_pre.shape}")

## 2. Regression Model Training & Evaluations

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Fit Linear Regression
lr = LinearRegression()
lr.fit(X_train_pre, y_train_reg)
lr_pred = lr.predict(X_test_pre)

print("--- Linear Regression Baseline Performance ---")
print(f"MAE: {mean_absolute_error(y_test_reg, lr_pred):.2f} mins")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, lr_pred)):.2f} mins")
print(f"R2 Score: {r2_score(y_test_reg, lr_pred):.4f}")

# 2. Fit Random Forest
rf = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42)
rf.fit(X_train_pre, y_train_reg)
rf_pred = rf.predict(X_test_pre)

print("\n--- Random Forest Regressor Performance ---")
print(f"MAE: {mean_absolute_error(y_test_reg, rf_pred):.2f} mins")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, rf_pred)):.2f} mins")
print(f"R2 Score: {r2_score(y_test_reg, rf_pred):.4f}")

## 3. Delay Classification Model Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

clf = RandomForestClassifier(n_estimators=80, max_depth=8, random_state=42)
clf.fit(X_train_pre, y_train_clf)
clf_pred = clf.predict(X_test_pre)

print("--- Delay Classifier Evaluation ---")
print(f"Accuracy Score: {accuracy_score(y_test_clf, clf_pred):.4f}")
print(f"F1 Score: {f1_score(y_test_clf, clf_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_clf, clf_pred))

## 4. SHAP Global Importance Concepts

Illustrating feature weighting in predicting deliver times.

In [ ]:
# Extract fitted Random Forest feature importance
ohe_cat_features = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols)
all_features = list(numerical_cols) + list(ohe_cat_features)

importances = rf.feature_importances_
df_importance = pd.DataFrame({
    "Feature": all_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("Top 10 features according to Random Forest Gini Importance:")
print(df_importance.head(10))